# Convolutionnal Neural Network

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import pandas as pd
from tokenizers import Tokenizer, models, trainers
from tokenizers.pre_tokenizers import Whitespace

from torchvision import datasets, transforms, utils

1. Reading the csv file

In [4]:
dataset = pd.read_csv('rotten_tomatoes_movie_reviews.csv')
for index, row in dataset.head(10).iterrows():
    print(f"Review: {row['reviewText']}")
    print(f"Sentiment: {row['scoreSentiment']}")
    print()

print(f"Number of reviews in the dataset: {len(dataset)}")

Review: Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water.
Sentiment: POSITIVE

Review: It doesn't matter if a movie costs 300 million or only 300 dollars; good is good and bad is bad, and Bloodmask: The Possession of Nicole Lameroux is just plain bad.
Sentiment: NEGATIVE

Review: The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped, but no live-action reference footage was used. The quality is due to the skill of the animators and Kodama's love for professional wrestling.
Sentiment: POSITIVE

Review: The film's out-of-touch attempts at humor may find them hunting for the reason the franchise was so popular in the first place.
Sentiment: NEGATIVE

Review: Its clumsy determination is endearing and sometimes wildly entertaining
Sentiment: POSITIVE

Review: With every new minute, there's another head-scratching choice that's boun

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


__Hyperparameters__

In [6]:
vocab_n = 10000
sequence_length = 64
batch_size = 4
num_workers = 0

__Dataset defining__

In [16]:
class SentimentDataset(data.Dataset):
    def __init__(self, root_dir, train=True):
        super().__init__()
        dataSet = pd.read_csv(root_dir)

        tokenizer = Tokenizer(models.BPE())
        tokenizer.pre_tokenizer = Whitespace()
        tokenizer.enable_padding(max_length=sequence_length)
        tokenizer.enable_truncation(max_length=sequence_length)
        tokenizer_trainer = trainers.BpeTrainer(vocab_size=vocab_n)


        data = []
        for index, row in dataSet.iterrows():
            if not isinstance(row['reviewText'], str): 
                continue
            if row['scoreSentiment'] == "POSITIVE":
                data.append((row['reviewText'], 1))
            elif row['scoreSentiment'] == "NEGATIVE":
                data.append((row['reviewText'], 0))
        

        train_size = int(0.8 * len(data))
        test_size = len(data) - train_size

        trainData = data[:train_size]
        textToTrainTO = []
        for text, label in trainData:
            textToTrainTO.append(text)

        for i in range(len(textToTrainTO)):
            if i > 10:
                break
            print(f"Review: {textToTrainTO[i]}")
            print()

        tokenizer.train_from_iterator(textToTrainTO, trainer=tokenizer_trainer)
        self.tokenizer = tokenizer

        self.data = data[:train_size] if train else data[train_size:]        

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoding = self.tokenizer.encode(text)

        return torch.tensor(encoding.ids), label



In [17]:
my_dataset = SentimentDataset('rotten_tomatoes_movie_reviews.csv', train=True)
my_loader = data.DataLoader(my_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

Review: Timed to be just long enough for most youngsters' brief attention spans -- and it's packed with plenty of interesting activity, both on land and under the water.

Review: It doesn't matter if a movie costs 300 million or only 300 dollars; good is good and bad is bad, and Bloodmask: The Possession of Nicole Lameroux is just plain bad.

Review: The choreography is so precise and lifelike at points one might wonder whether the movie was rotoscoped, but no live-action reference footage was used. The quality is due to the skill of the animators and Kodama's love for professional wrestling.

Review: The film's out-of-touch attempts at humor may find them hunting for the reason the franchise was so popular in the first place.

Review: Its clumsy determination is endearing and sometimes wildly entertaining

Review: With every new minute, there's another head-scratching choice that's bound to elicit some amazing out-loud responses, so this feels like a true party flick.

Review: Emotion

In [21]:
encodings, labels = next(iter(my_loader))

for encoding, label in zip(encodings, labels):
    print(f"Encoding: {encoding}")
    print(f"text: {my_dataset.tokenizer.decode(encoding.tolist())}")
    print(f"Labels: {label}")

Encoding: tensor([3350,  243, 6057,  195,  188, 9952,  716,  369,  595, 8094,  730,  413,
          65,  291, 7619,  298, 1900,  345,   65, 8184,  195,   65,   76,  211,
         220,  337, 4317,  195, 2708, 3175,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
text: Perhaps The Wolf and the Lion should have been stripped down into a movie excl us ively about a wolf and a l ion for more meaningful and affecting results
Labels: 0
Encoding: tensor([ 795,  188, 1040,  220,   65,  291,   12, 6441, 4997, 1693,  456,  412,
         672, 4142,  303,  349, 5684, 4997,   14,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0, 